In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense
)

from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

print("TensorFlow version:", tf.__version__)

I0000 00:00:1786191914.140480  171627 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786191914.244947  171627 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786191917.284090  171627 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0


In [6]:
data = pd.read_csv(
    "pjm_processed.csv",
    parse_dates=["timestamp"]
)

In [7]:
data["timestamp"] = pd.to_datetime(data["timestamp"])

data = data.sort_values("timestamp")

data = data.set_index("timestamp")

In [8]:
print(data["demand"].describe())

count    145224.000000
mean      32078.418306
std        6466.681191
min       14544.000000
25%       27569.000000
50%       31419.000000
75%       35647.000000
max       62009.000000
Name: demand, dtype: float64


In [9]:
print("Missing demand:", data["demand"].isna().sum())

Missing demand: 0


In [10]:
print(
    data.index.to_series().diff().value_counts().head()
)

timestamp
0 days 01:00:00    145223
Name: count, dtype: int64


In [11]:
target = data[["demand"]].copy()

In [12]:
test_size = int(len(target) * 0.20)

test_start = len(target) - test_size

remaining = target.iloc[:test_start]
test = target.iloc[test_start:]

In [13]:
validation_hours = 30 * 24

val_start = len(remaining) - validation_hours

train = remaining.iloc[:val_start]
validation = remaining.iloc[val_start:]

In [14]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (115460, 1)
Validation: (720, 1)
Test: (29044, 1)


In [15]:
print(
    "Train:",
    train.index.min(),
    "→",
    train.index.max()
)

print(
    "Validation:",
    validation.index.min(),
    "→",
    validation.index.max()
)

print(
    "Test:",
    test.index.min(),
    "→",
    test.index.max()
)

Train: 2002-01-08 01:00:00 → 2015-03-11 20:00:00
Validation: 2015-03-11 21:00:00 → 2015-04-10 20:00:00
Test: 2015-04-10 21:00:00 → 2018-08-03 00:00:00


In [16]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(
    train[["demand"]]
)

validation_scaled = scaler.transform(
    validation[["demand"]]
)

test_scaled = scaler.transform(
    test[["demand"]]
)

In [17]:
def create_sequences(
    data,
    input_length,
    output_length
):

    X = []
    y = []

    for i in range(
        len(data) - input_length - output_length + 1
    ):

        X.append(
            data[
                i:i + input_length
            ]
        )

        y.append(
            data[
                i + input_length:
                i + input_length + output_length
            ]
        )

    return np.array(X), np.array(y)

In [18]:
def prepare_sequences(
    sequence_length,
    forecast_horizon=24
):

    # Training sequences
    X_train, y_train = create_sequences(
        train_scaled,
        sequence_length,
        forecast_horizon
    )

    # Validation needs historical context
    val_input = np.concatenate([
        train_scaled[-sequence_length:],
        validation_scaled
    ])

    X_val, y_val = create_sequences(
        val_input,
        sequence_length,
        forecast_horizon
    )

    # Test needs historical context
    test_input = np.concatenate([
        validation_scaled[-sequence_length:],
        test_scaled
    ])

    X_test, y_test = create_sequences(
        test_input,
        sequence_length,
        forecast_horizon
    )

    # Remove final dimension from y
    y_train = y_train.reshape(
        y_train.shape[0],
        y_train.shape[1]
    )

    y_val = y_val.reshape(
        y_val.shape[0],
        y_val.shape[1]
    )

    y_test = y_test.reshape(
        y_test.shape[0],
        y_test.shape[1]
    )

    return (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test
    )

In [19]:
X_train, y_train, X_val, y_val, X_test, y_test = prepare_sequences(
    sequence_length=24,
    forecast_horizon=24
)

In [20]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (115413, 24, 1)
y_train: (115413, 24)
X_val: (697, 24, 1)
y_val: (697, 24)
X_test: (29021, 24, 1)
y_test: (29021, 24)


In [21]:
#Rnn
def build_rnn(sequence_length):

    model = Sequential([
        SimpleRNN(
            64,
            input_shape=(sequence_length, 1)
        ),

        Dense(24)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

In [22]:
#LSTM
def build_lstm(sequence_length):

    model = Sequential([
        LSTM(
            64,
            input_shape=(sequence_length, 1)
        ),

        Dense(24)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

In [23]:
#GRU
def build_gru(sequence_length):

    model = Sequential([
        GRU(
            64,
            input_shape=(sequence_length, 1)
        ),

        Dense(24)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

In [24]:
#BILSTm
def build_bilstm(sequence_length):

    model = Sequential([
        Bidirectional(
            LSTM(64),
            input_shape=(sequence_length, 1)
        ),

        Dense(24)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

In [28]:
def evaluate_forecast(y_true, y_pred):

    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    mae = mean_absolute_error(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mape = np.mean(
        np.abs((y_true - y_pred) / y_true)
    ) * 100

    r2 = r2_score(y_true, y_pred)

    bias = np.mean(y_pred - y_true)

    return mae, mse, rmse, mape, r2, bias

In [29]:
def inverse_scale(predictions):

    original_shape = predictions.shape

    predictions = predictions.reshape(-1, 1)

    predictions = scaler.inverse_transform(
        predictions
    )

    return predictions.reshape(
        original_shape
    )

In [31]:
sequence_lengths = [24, 48, 168]
forecast_horizon = 24
all_results = []

In [32]:

for sequence_length in sequence_lengths:

    print("\n" + "=" * 60)
    print(
        f"SEQUENCE: {sequence_length} HOURS → 24 HOURS"
    )
    print("=" * 60)

    # Prepare sequences
    (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test
    ) = prepare_sequences(
        sequence_length=sequence_length,
        forecast_horizon=24
    )

    # ------------------------------------------------
    # RNN
    # ------------------------------------------------

    print("\nTraining RNN...")

    rnn_model = build_rnn(sequence_length)

    rnn_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=32,
        
        verbose=1
    )

    rnn_pred = rnn_model.predict(
        X_test,
        verbose=0
    )

    rnn_pred = inverse_scale(rnn_pred)
    y_actual = inverse_scale(y_test)

    metrics = evaluate_forecast(
        y_actual,
        rnn_pred
    )

    all_results.append({
        "Sequence": sequence_length,
        "Horizon": 24,
        "Model": "RNN",
        "MAE": metrics[0],
        "MSE": metrics[1],
        "RMSE": metrics[2],
        "MAPE": metrics[3],
        "R2": metrics[4],
        "Bias": metrics[5]
    })

    # ------------------------------------------------
    # LSTM
    # ------------------------------------------------

    print("\nTraining LSTM...")

    lstm_model = build_lstm(sequence_length)

    lstm_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=32,
        
        verbose=1
    )

    lstm_pred = lstm_model.predict(
        X_test,
        verbose=0
    )

    lstm_pred = inverse_scale(lstm_pred)

    metrics = evaluate_forecast(
        y_actual,
        lstm_pred
    )

    all_results.append({
        "Sequence": sequence_length,
        "Horizon": 24,
        "Model": "LSTM",
        "MAE": metrics[0],
        "MSE": metrics[1],
        "RMSE": metrics[2],
        "MAPE": metrics[3],
        "R2": metrics[4],
        "Bias": metrics[5]
    })

    # ------------------------------------------------
    # GRU
    # ------------------------------------------------

    print("\nTraining GRU...")

    gru_model = build_gru(sequence_length)

    gru_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=32,
        
        verbose=1
    )

    gru_pred = gru_model.predict(
        X_test,
        verbose=0
    )

    gru_pred = inverse_scale(gru_pred)

    metrics = evaluate_forecast(
        y_actual,
        gru_pred
    )

    all_results.append({
        "Sequence": sequence_length,
        "Horizon": 24,
        "Model": "GRU",
        "MAE": metrics[0],
        "MSE": metrics[1],
        "RMSE": metrics[2],
        "MAPE": metrics[3],
        "R2": metrics[4],
        "Bias": metrics[5]
    })

    # ------------------------------------------------
    # Bi-LSTM
    # ------------------------------------------------

    print("\nTraining Bi-LSTM...")

    bilstm_model = build_bilstm(sequence_length)

    bilstm_model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=32,
        
        verbose=1
    )

    bilstm_pred = bilstm_model.predict(
        X_test,
        verbose=0
    )

    bilstm_pred = inverse_scale(bilstm_pred)

    metrics = evaluate_forecast(
        y_actual,
        bilstm_pred
    )

    all_results.append({
        "Sequence": sequence_length,
        "Horizon": 24,
        "Model": "Bi-LSTM",
        "MAE": metrics[0],
        "MSE": metrics[1],
        "RMSE": metrics[2],
        "MAPE": metrics[3],
        "R2": metrics[4],
        "Bias": metrics[5]
    })


SEQUENCE: 24 HOURS → 24 HOURS

Training RNN...
Epoch 1/15


E0000 00:00:1786192991.292137  171627 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3607/3607 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0040 - val_loss: 0.0022
Epoch 2/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.0028 - val_loss: 0.0023
Epoch 3/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0027 - val_loss: 0.0021
Epoch 4/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0027 - val_loss: 0.0021
Epoch 5/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0026 - val_loss: 0.0023
Epoch 6/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0026 - val_loss: 0.0021
Epoch 7/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0026 - val_loss: 0.0022
Epoch 8/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0026 - val_loss: 0.0021
Epoch 9/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - loss: 0.0025 - val_loss: 0.0022
Epoch 10/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0025 - val_loss: 0.0021
Epoch 11/15
3607/3607 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0025 - val_loss: 0.0022
Epoch 12/15
3607/3607 ━━━━━━━━

3606/3606 ━━━━━━━━━━━━━━━━━━━━ 25s 7ms/step - loss: 0.0022 - val_loss: 0.0020
Epoch 6/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 28s 8ms/step - loss: 0.0022 - val_loss: 0.0018
Epoch 7/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 26s 7ms/step - loss: 0.0021 - val_loss: 0.0021
Epoch 8/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 0.0020 - val_loss: 0.0018
Epoch 9/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 0.0019 - val_loss: 0.0021
Epoch 10/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 23s 6ms/step - loss: 0.0019 - val_loss: 0.0018
Epoch 11/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 0.0019 - val_loss: 0.0018
Epoch 12/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 0.0019 - val_loss: 0.0017
Epoch 13/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 0.0018 - val_loss: 0.0017
Epoch 14/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 0.0018 - val_loss: 0.0017
Epoch 15/15
3606/3606 ━━━━━━━━━━━━━━━━━━━━ 25s 7ms/step - loss: 0.0018 - val_loss: 0.0018

Training LSTM...
Epoch 1/

3603/3603 ━━━━━━━━━━━━━━━━━━━━ 71s 20ms/step - loss: 0.0019 - val_loss: 0.0016
Epoch 10/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 65s 18ms/step - loss: 0.0019 - val_loss: 0.0020
Epoch 11/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 65s 18ms/step - loss: 0.0018 - val_loss: 0.0017
Epoch 12/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 65s 18ms/step - loss: 0.0018 - val_loss: 0.0016
Epoch 13/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 65s 18ms/step - loss: 0.0059 - val_loss: 0.0266
Epoch 14/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 65s 18ms/step - loss: 0.0042 - val_loss: 0.0025
Epoch 15/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 65s 18ms/step - loss: 0.0028 - val_loss: 0.0024

Training LSTM...
Epoch 1/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 170s 47ms/step - loss: 0.0046 - val_loss: 0.0025
Epoch 2/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 162s 45ms/step - loss: 0.0024 - val_loss: 0.0020
Epoch 3/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 162s 45ms/step - loss: 0.0020 - val_loss: 0.0018
Epoch 4/15
3603/3603 ━━━━━━━━━━━━━━━━━━━━ 161s 45ms/step - loss: 0.0018 - val_loss: 

In [33]:
print(all_results)

[{'Sequence': 24, 'Horizon': 24, 'Model': 'RNN', 'MAE': 1685.8607670499514, 'MSE': 5500636.063154633, 'RMSE': np.float64(2345.343485111431), 'MAPE': np.float64(5.462425126549631), 'R2': 0.8693303456911676, 'Bias': np.float64(238.7567408840931)}, {'Sequence': 24, 'Horizon': 24, 'Model': 'LSTM', 'MAE': 1512.9144772807972, 'MSE': 4690809.336097136, 'RMSE': np.float64(2165.8276330532713), 'MAPE': np.float64(4.8964830673500455), 'R2': 0.8885680806112213, 'Bias': np.float64(324.47417218316855)}, {'Sequence': 24, 'Horizon': 24, 'Model': 'GRU', 'MAE': 1488.7609497727894, 'MSE': 4550922.688782145, 'RMSE': np.float64(2133.2891713928857), 'MAPE': np.float64(4.805768566576084), 'R2': 0.8918911399151281, 'Bias': np.float64(246.61809295681988)}, {'Sequence': 24, 'Horizon': 24, 'Model': 'Bi-LSTM', 'MAE': 1523.322923964554, 'MSE': 4654855.798063688, 'RMSE': np.float64(2157.5114827188495), 'MAPE': np.float64(4.937271441717121), 'R2': 0.8894221702714116, 'Bias': np.float64(149.5863314695496)}, {'Sequenc

In [34]:
import pandas as pd

results_df = pd.DataFrame(all_results)

results_df

,Sequence,Horizon,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,24,24,RNN,1685.860767,5.500636e+06,2345.343485,5.462425,0.869330,238.756741
1,24,24,LSTM,1512.914477,4.690809e+06,2165.827633,4.896483,0.888568,324.474172
2,24,24,GRU,1488.760950,4.550923e+06,2133.289171,4.805769,0.891891,246.618093
3,24,24,Bi-LSTM,1523.322924,4.654856e+06,2157.511483,4.937271,0.889422,149.586331
4,48,24,RNN,1566.645508,4.637170e+06,2153.408885,5.001122,0.889842,60.842667
5,48,24,LSTM,1417.016894,4.063550e+06,2015.824985,4.559036,0.903469,307.211975
6,48,24,GRU,1407.372058,3.985015e+06,1996.250119,4.497683,0.905334,91.033141
7,48,24,Bi-LSTM,1414.019266,4.042415e+06,2010.575733,4.513480,0.903971,112.758116
8,168,24,RNN,1894.869473,6.283783e+06,2506.747479,6.203033,0.850726,388.770340
9,168,24,LSTM,1356.110610,3.728263e+06,1930.871173,4.324987,0.911434,267.204871


In [35]:
best_model = results_df.loc[results_df['RMSE'].idxmin()]

print(best_model)

Sequence               168
Horizon                 24
Model                  GRU
MAE            1364.036036
MSE         3715290.876289
RMSE           1927.508982
MAPE              4.343224
R2                0.911742
Bias            331.006106
Name: 10, dtype: object


In [36]:
results_df.to_csv("forecast_results.csv", index=False)

# phase_06 MODEL IMPROVEMENT